In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the saved train/test data
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

# Define day columns
train_days = [f'd_{i}' for i in range(1, 1914)]  # d_1 to d_1913
test_days = [f'd_{i}' for i in range(1914, 1942)]  # d_1914 to d_1941

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

Training data shape: (1437, 1915)
Test data shape: (1437, 30)


In [3]:
def moving_average_forecast(train_data, test_days, window=7):
    """
    Forecast using average of last N days (moving average)
    
    Args:
        train_data: Training data with historical sales
        test_days: List of test day columns to forecast
        window: Number of days to average (default: 7)
    
    Returns:
        Dictionary of forecasts for each product
    """
    forecasts = {}
    
    for idx, row in train_data.iterrows():
        product_id = row['id']
        forecast = []
        
        for i, test_day in enumerate(test_days):
            day_num = int(test_day.split('_')[1])
            
            # Get average of last 'window' days before this test day
            start_day = max(1, day_num - window)
            end_day = day_num - 1
            
            window_days = [f'd_{j}' for j in range(start_day, end_day + 1)]
            window_days = [d for d in window_days if d in train_data.columns]
            
            if window_days:
                avg_sales = row[window_days].mean()
                forecast.append(avg_sales)
            else:
                forecast.append(0)
        
        forecasts[product_id] = forecast
    
    return forecasts

In [4]:
# Get actual values for evaluation
actuals = {}
for idx, row in test_data.iterrows():
    product_id = row['id']
    actuals[product_id] = row[test_days].tolist()

# Test different window sizes
results = {}
for window in [7, 14, 21, 28]:
    print(f"Testing {window}-day moving average...")
    ma_forecasts = moving_average_forecast(train_data, test_days, window)
    
    # Calculate WAPE
    total_absolute_error = 0
    total_actual_sales = 0
    
    for product_id in ma_forecasts.keys():
        actual = actuals[product_id]
        forecast = ma_forecasts[product_id]
        
        for i in range(len(actual)):
            total_absolute_error += abs(actual[i] - forecast[i])
            total_actual_sales += actual[i]
    
    if total_actual_sales > 0:
        wape = total_absolute_error / total_actual_sales
        results[window] = wape
        print(f"  {window}-day MA WAPE: {wape:.3f} ({wape:.1%})")
    else:
        print(f"  {window}-day MA WAPE: undefined (zero actual sales)")

print("\nSummary:")
for window, wape in sorted(results.items()):
    print(f"  {window}-day Moving Average: {wape:.3f} ({wape:.1%})")

Testing 7-day moving average...
  7-day MA WAPE: 0.932 (93.2%)
Testing 14-day moving average...
  14-day MA WAPE: 0.846 (84.6%)
Testing 21-day moving average...
  21-day MA WAPE: 0.762 (76.2%)
Testing 28-day moving average...
  28-day MA WAPE: 0.685 (68.5%)

Summary:
  7-day Moving Average: 0.932 (93.2%)
  14-day Moving Average: 0.846 (84.6%)
  21-day Moving Average: 0.762 (76.2%)
  28-day Moving Average: 0.685 (68.5%)


In [5]:
# Compare with seasonal naive baseline
seasonal_naive_wape = 0.944  # From your previous results

print("Baseline Comparison:")
print(f"Seasonal Naive (7-day lag): {seasonal_naive_wape:.3f} ({seasonal_naive_wape:.1%})")
for window, wape in sorted(results.items()):
    improvement = seasonal_naive_wape - wape
    print(f"{window}-day Moving Average: {wape:.3f} ({wape:.1%}) - Improvement: {improvement:+.1%}")

Baseline Comparison:
Seasonal Naive (7-day lag): 0.944 (94.4%)
7-day Moving Average: 0.932 (93.2%) - Improvement: +1.2%
14-day Moving Average: 0.846 (84.6%) - Improvement: +9.8%
21-day Moving Average: 0.762 (76.2%) - Improvement: +18.2%
28-day Moving Average: 0.685 (68.5%) - Improvement: +25.9%
